# CARec encoder and dimension study

Load the verified Amazon Electronics catalogue and fixed interaction partitions, then compare item-text encoders and embedding dimensions. Warm/cold Recall@10 and NDCG@10 will measure recommendation quality.

Data exploration, original CARec text preparation, sampling, splitting, and verification live in [data_exploration.ipynb](data_exploration.ipynb). This notebook reads those saved outputs and does not prepare the dataset again.

Use the `NI-PML benchmarks` kernel. References: [CARec paper](https://doi.org/10.1145/3627673.3679535), [pinned source](https://github.com/ChenMetanoia/CARec/tree/bbfc7f385e796840f6eb5c967dfc8714284958ce).


## Imports and device

Select the `NI-PML benchmarks` kernel from the project `.venv`. It contains the pinned dependencies. Use Apple's MPS backend when available; the device fallbacks also allow CPU execution.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

mps_backend = getattr(torch.backends, "mps", None)
if mps_backend is not None and mps_backend.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


## Load the prepared dataset

Read the verified manifest, check the saved input hashes, and load product text, the four interaction partitions, and ID maps. The raw JSONL files are not used. The known duplicate-pair and split-rounding findings remain recorded in `manifest["audit"]`.


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError(
        "Select the 'NI-PML benchmarks' kernel or .venv/bin/python, then restart and run the notebook. "
        f"Current Python: {sys.executable}"
    )

PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
if not (PREPARED_DIR / "manifest.json").is_file():
    raise FileNotFoundError("Prepare the data in data_exploration.ipynb first.")
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


In [3]:
import gzip
import hashlib
import json

manifest = json.loads((PREPARED_DIR / "manifest.json").read_text())
if manifest["status"] != "verified":
    raise ValueError("Run the verification cells in data_exploration.ipynb first.")

partition_names = ("train", "valid", "test", "cold")
files_to_load = ["items.jsonl.gz", "user_ids.tsv.gz", "item_ids.tsv.gz"]
files_to_load += [f"{name}.tsv.gz" for name in partition_names]
for name in files_to_load:
    path = PREPARED_DIR / name
    expected = manifest["files"][name]
    with path.open("rb") as source:
        assert hashlib.file_digest(source, "sha256").hexdigest() == expected["sha256"], name
    assert path.stat().st_size == expected["bytes"], name

with gzip.open(PREPARED_DIR / "items.jsonl.gz", "rt", encoding="utf-8") as source:
    items = pd.DataFrame(json.loads(line) for line in source)
partitions = {
    name: pd.read_csv(
        PREPARED_DIR / f"{name}.tsv.gz", sep="\t",
        dtype={"user_id:token": str, "item_id:token": str}, float_precision="round_trip",
    )
    for name in partition_names
}
train, valid, test, cold = (partitions[name] for name in partition_names)
user_map = pd.read_csv(PREPARED_DIR / "user_ids.tsv.gz", sep="\t", dtype={"user_id": str})
item_map = pd.read_csv(PREPARED_DIR / "item_ids.tsv.gz", sep="\t", dtype={"item_id": str})
assert items["item_id:token"].is_unique
assert len(items) == manifest["catalogue_products"]
assert item_map["item_id"].tolist() == items["item_id:token"].tolist()
assert len(user_map) == train["user_id:token"].nunique()
for name, frame in partitions.items():
    assert len(frame) == manifest["counts"][name]["rows"], name
print(f"Loaded {len(items):,} products and {len(user_map):,} users")
print(pd.DataFrame(manifest["counts"]).T.to_string())
display(items[["item_id:token", "text"]].head(2))


Loaded 34,220 products and 81,520 users
         rows  users  products
train  452606  81520     32416
valid   14853  12386      9322
test   133274  81520     28515
cold    23163  18680      1797


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...


## Product text and encoder inputs

Product text is loaded unchanged from `items.jsonl.gz`. The preparation notebook generated it with the pinned CARec `RawDataProcessor._preprocess_text_feature` and Electronics settings. Product IDs stay separate from encoder inputs. The examples below format two loaded products; no encoder runs yet.

CARec uses a frozen text encoder to generate item vectors before recommender training. It then learns user representations and an MLP that aligns item vectors with interaction data. The text encoder itself is not fine-tuned. See the pinned [preprocessing](https://github.com/ChenMetanoia/CARec/blob/bbfc7f385e796840f6eb5c967dfc8714284958ce/data/utils.py) and [recommender](https://github.com/ChenMetanoia/CARec/blob/bbfc7f385e796840f6eb5c967dfc8714284958ce/model/collabcontex.py).

These are our six planned checkpoints, pending inference pilots. `text` below is always the same product text.

| Encoder | Prepared input |
| --- | --- |
| [MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | Plain text |
| [MPNet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | Plain text |
| [E5-base-v2](https://huggingface.co/intfloat/e5-base-v2) | `query: {text}`, a formatting prefix rather than a task instruction |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | `Instruct: {instruction}\nQuery:{text}` and plain text |
| [Multilingual E5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct) | `Instruct: {instruction}\nQuery: {text}` and plain text |
| [INSTRUCTOR-base](https://huggingface.co/hkunlp/instructor-base) | `[instruction, text]` and `["", text]` through its instruction-aware API |

We prepare nine input conditions without loading any model. The shared recommendation instruction below is a working choice for the pilot. The no-instruction E5-instruct and empty-instruction INSTRUCTOR conditions still need compatibility checks. Later encoding must disable automatic prompts to preserve these exact inputs. Run the saved-data loading cells first. Full catalogue text is available in `items`; the input examples use its first two products.


In [4]:
product_text_preview = items[["item_id:token", "text"]].head(2).rename(
    columns={"item_id:token": "asin"}
).to_dict("records")

TASK_INSTRUCTION = "Represe nt this product for personalized recommendation."
item_ids = [item["asin"] for item in product_text_preview]
item_texts = [item["text"] for item in product_text_preview]

encoder_inputs = {
    ("sentence-transformers/all-MiniLM-L6-v2", "standard"): item_texts.copy(),
    ("sentence-transformers/all-mpnet-base-v2", "standard"): item_texts.copy(),
    ("intfloat/e5-base-v2", "standard"): [f"query: {text}" for text in item_texts],
    ("Qwen/Qwen3-Embedding-0.6B", "with_instruction"): [
        f"Instruct: {TASK_INSTRUCTION}\nQuery:{text}" for text in item_texts
    ],
    ("Qwen/Qwen3-Embedding-0.6B", "without_instruction"): item_texts.copy(),
    ("intfloat/multilingual-e5-large-instruct", "with_instruction"): [
        f"Instruct: {TASK_INSTRUCTION}\nQuery: {text}" for text in item_texts
    ],
    ("intfloat/multilingual-e5-large-instruct", "without_instruction"): item_texts.copy(),
    ("hkunlp/instructor-base", "with_instruction"): [
        [TASK_INSTRUCTION + " ", text] for text in item_texts
    ],
    ("hkunlp/instructor-base", "without_instruction"): [["", text] for text in item_texts],
}

for (checkpoint, prompt_mode), inputs in encoder_inputs.items():
    print(f"{checkpoint} / {prompt_mode} / {len(inputs)} products")
    if inputs:
        print(inputs[0])
    print()


sentence-transformers/all-MiniLM-L6-v2 / standard / 2 products
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

sentence-transformers/all-mpnet-base-v2 / standard / 2 products
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

intfloat/e5-base-v2 / standard / 2 products
query: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

Qwen/Qwen3-Embedding-0.6B / with_instruction / 2 products
Instruct: Represe nt this product for personalized recommendation.
Query:Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

Qwen/Qwen3-Embedding-0.6B / without_instruction / 2 products
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

intfloat/multilingual-e5-large-instruct / with_instruction / 2 products
Instruct: Represe nt this product for personalized recommendation.
Query: Elephant Run Visit Amazon's R

## CARec imports

Load the pinned model and evaluation helpers for later experiments. These imports do not start training.


In [7]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.trainer import Trainer

carec = load_carec()
CollabContex = carec.CollabContex
ColdEvaluator = carec.ColdEvaluator
data_preparation = carec.data_preparation

print(f"CARec: {carec.commit}")

CARec: bbfc7f385e796840f6eb5c967dfc8714284958ce
